# 13: Ensemble Learning

**Author:** Dr Rob Lyon

**Version:** 1.0

## Code & License
Copyright © 2026 Robert Lyon. All Rights Reserved.

This notebook and all associated materials are the intellectual property of the author.

Permission is granted solely to read, study, and analyse this material for personal educational purposes. No other rights are granted.

Without the prior written consent of the author, you may not:

* Copy, reproduce, redistribute, publish, transmit, or display this work in whole or in part.
* Modify, adapt, transform, translate, or create derivative works based on this material.
* Incorporate any portion of this work into another project, publication, product, model, dataset, or codebase.
* Use this material for commercial purposes.
* Remove or alter this copyright notice.

All intellectual property rights, including copyright and any derivative rights, remain exclusively vested in the author.

Access to this material does not constitute a transfer of ownership, license, or any other intellectual property rights except as expressly stated above.

---

**Note:**

This notebook is designed to be viewed from within JupyterLab. The Table of Contents and "Back to Table of Contents" links rely on JupyterLab's anchor handling to jump between sections.

If you open this notebook in another tool, such as PyCharm or Visual Studio Code, these anchor links may not work as expected. You can still navigate the notebook in those tools by using the headings directly, most editors provide an outline or contents panel that lists them for you.

---

## Table of Contents

1. [Introduction](#1.-Introduction)
2. [Useful Links](#2.-Useful-Links)
3. [Libraries and Environment Setup](#3.-Libraries-and-Environment-Setup)
4. [The Wisdom of the Crowd](#4.-The-Wisdom-of-the-Crowd)
5. [Bagging: Bootstrap Aggregating](#5.-Bagging:-Bootstrap-Aggregating)
    - [5.1 The Bootstrap](#5.1-The-Bootstrap)
    - [5.2 How Bagging Reduces Variance](#5.2-How-Bagging-Reduces-Variance)
    - [5.3 Out-of-Bag Evaluation](#5.3-Out-of-Bag-Evaluation)
    - [5.4 Bagging with scikit-learn](#5.4-Bagging-with-scikit-learn)
    - [5.5 Bagging on the Wisconsin Dataset](#5.5-Bagging-on-the-Wisconsin-Dataset)
6. [Boosting: Sequential Error Correction](#6.-Boosting:-Sequential-Error-Correction)
    - [6.1 The Core Idea](#6.1-The-Core-Idea)
    - [6.2 Bagging vs Boosting](#6.2-Bagging-vs-Boosting)
7. [AdaBoost](#7.-AdaBoost)
    - [7.1 The Algorithm in Detail](#7.1-The-Algorithm-in-Detail)
    - [7.2 The Exponential Loss Connection](#7.2-The-Exponential-Loss-Connection)
    - [7.3 AdaBoost with scikit-learn](#7.3-AdaBoost-with-scikit-learn)
    - [7.4 AdaBoost on the Wisconsin Dataset](#7.4-AdaBoost-on-the-Wisconsin-Dataset)
    - [7.5 Learning Rate and the n_estimators Trade-off](#7.5-Learning-Rate-and-the-n_estimators-Trade-off)
8. [Comparing Bagging and Boosting Side by Side](#8.-Comparing-Bagging-and-Boosting-Side-by-Side)
9. [Summary](#9.-Summary)
10. [References](#10.-References)

---

## 1. Introduction

Most of the classifiers we have built so far in this series, decision trees, Naïve Bayes, SVMs, have all been single-model approaches: train one model, apply it. Each has its own strengths and failure modes, and a recurring theme has been the tension between **bias and variance**: a model that is too simple cannot capture the structure in the data (high bias); a model that is too complex fits the training data so closely that it generalises poorly to new examples (high variance).

**Ensemble methods** take a different route. Rather than trying to build one perfect model, they build many models and combine their predictions. The intuition is that independent models tend to make different mistakes, and when you average or aggregate those mistakes together, the errors cancel out. What remains is something more reliable than any single model could produce on its own. By now you've become familiar with one such ensemble method, the Random Forest, but it is not the only approach.

This notebook covers the two foundational ensemble strategies:

- **Bagging** (Bootstrap Aggregating), introduced by Breiman (1996), which trains many independent copies of the same model on different random subsets of the training data and combines their predictions by majority vote or averaging.
- **Boosting**, specifically **AdaBoost** (Freund & Schapire, 1997), which trains a sequence of weak models, each one focusing on the examples that previous models got wrong, and combines them into a single, progressively stronger classifier.

Both methods are applied throughout to the **Wisconsin Diagnostic Breast Cancer dataset**, a real clinical dataset where the task is to classify tumours as benign or malignant from 30 numerical features derived from cell nucleus images. This makes the examples medically concrete and the performance numbers directly interpretable.

### Learning Objectives

By the end of this notebook you should be able to:

- Explain the bias-variance decomposition and how ensemble methods address it.
- Describe bootstrap sampling and explain why it introduces useful diversity between ensemble members.
- Explain how bagging reduces variance without increasing bias, and describe out-of-bag evaluation.
- Apply `BaggingClassifier` in scikit-learn, inspect its key hyperparameters and fitted attributes.
- Describe the sequential, adaptive nature of boosting and contrast it with bagging.
- Walk through the AdaBoost algorithm step by step: sample reweighting, weak learner weight $\alpha_t$, and the final weighted vote.
- Apply `AdaBoostClassifier` in scikit-learn, explain the role of `learning_rate` and `n_estimators`, and interpret the results on the Wisconsin dataset.
- Compare bagging and boosting on the same dataset and reason about when each is preferable.

---

---

## 2. Useful Links

| Link | Description |
|------|-------------|
| [Breiman (1996), Bagging Predictors](https://doi.org/10.1007/BF00058655) | The original bagging paper, introducing bootstrap aggregating as a variance-reduction method for unstable classifiers. |
| [Freund & Schapire (1997), A Decision-Theoretic Generalization of On-Line Learning and an Application to Boosting](https://doi.org/10.1006/jcss.1997.1504) | The paper introducing AdaBoost. The algorithm, its weight-update rule, and the theoretical guarantee on training error are all derived here. |
| [scikit-learn: BaggingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html) | Full API reference for `BaggingClassifier`. |
| [scikit-learn: AdaBoostClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html) | Full API reference for `AdaBoostClassifier`. |
| [scikit-learn: Ensemble Methods user guide](https://scikit-learn.org/stable/modules/ensemble.html) | Conceptual overview of all ensemble methods in scikit-learn, including bagging, boosting, random forests, and stacking. |
| [UCI Wisconsin Breast Cancer (Diagnostic) Dataset](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic) | The original UCI repository entry for the dataset used in this notebook. |

---

---

## 3. Libraries and Environment Setup

### 3.1 Working Environment

To run the code in this notebook, you will need a Python environment with the required libraries listed below installed.

The easiest way to get started is to use the **Project IO** Docker container, which provides a fully configured and tested environment containing all required Python packages and supporting dependencies. Instructions for downloading and running the container can be found in the **project-io** GitHub repository:

https://github.com/scienceguyrob/project-io

Using the Docker container ensures that the notebooks run in exactly the same environment in which they were developed and tested.

If you prefer not to use Docker, a good alternative is to install the [Anaconda Distribution](https://www.anaconda.com/products/distribution). You can then create a new Python environment and install the required libraries using either `conda install` or `pip install`.

### 3.2 Libraries Used in This Notebook

| Library | Purpose | Documentation |
|---------|---------|---------------|
| **NumPy** (`numpy`) | Numerical arrays and mathematical operations. | [numpy.org](https://numpy.org/doc/stable/) |
| **Pandas** (`pandas`) | Tabular data handling and summary statistics. | [pandas.pydata.org](https://pandas.pydata.org/docs/) |
| **Matplotlib** (`matplotlib.pyplot`) | All plots and figures. | [matplotlib.org](https://matplotlib.org/stable/) |
| **scikit-learn** (`sklearn`) | Dataset loading, classifiers, model selection, and evaluation metrics. | [scikit-learn.org](https://scikit-learn.org/stable/) |

### 3.3 Imports

All library imports for this notebook are placed in the single cell below. This is deliberate best practice: keeping all imports in one place makes it easy to see at a glance what is required and avoids confusing `NameError` exceptions that arise when an import cell is accidentally skipped.

> ⚠️ **You must run the cell below before executing any other code cell in this notebook.**

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# All library imports for this notebook are placed here for clarity.
# Run this cell first before executing any code.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

# scikit-learn: dataset
from sklearn.datasets import load_breast_cancer

# scikit-learn: model selection and preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# scikit-learn: base estimators
from sklearn.tree import DecisionTreeClassifier

# scikit-learn: ensemble methods
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier

# scikit-learn: evaluation
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, RocCurveDisplay, recall_score,
)

import matplotlib
matplotlib.rcParams['figure.max_open_warning'] = 50

# Shared random state for reproducibility across all cells
RANDOM_STATE = 42

print('Libraries loaded successfully.')

---

## 4. The Wisdom of the Crowd

🔙 [Back to Table of Contents](#Table-of-Contents)

Before diving into the mechanics of any particular ensemble algorithm, it is worth understanding *why* combining models should help at all. The core argument is statistical, and it rests on the **bias-variance decomposition**.

Recall from the earlier notebooks in this series that the expected prediction error of any model on unseen data can be decomposed into three sources:

$$\text{Expected error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible noise}$$

- **Bias** measures how far the model's average prediction is from the true answer. A biased model is systematically wrong in a consistent direction, because its family of possible functions is too restricted to capture the true relationship in the data.
- **Variance** measures how much the model's predictions change when it is trained on different subsets of the data. A high-variance model is very sensitive to the specific training examples it saw, and tends to overfit.
- **Irreducible noise** is the randomness inherent in the data itself, which no model can remove.

Different algorithms sit at different points on this bias-variance spectrum. Decision trees (particularly deep, unpruned ones) tend to have **low bias but high variance**: they can fit complex patterns, but their predictions change substantially depending on which training examples they happen to see. Simple models such as a decision stump (a single-node tree) tend to have **high bias but low variance**: they are consistently wrong in the same way regardless of the training data.

Ensemble methods address this in two distinct ways:

- **Bagging** targets **variance**. It builds many independent high-variance models on different random subsets of the training data, and then averages their predictions. Because the models are trained on different data, their errors are at least partially decorrelated. When you average many decorrelated errors together, the variance drops, while the bias stays roughly the same.
- **Boosting** targets **bias**. It builds a sequence of simple, low-variance models (called **weak learners**) and combines them in a way that specifically corrects the mistakes of previous models. Each weak learner individually has high bias, but the sequential correction process drives the bias down, producing a final ensemble with both lower bias and reasonable variance.

### An Analogy: Polling

A useful analogy is polling. If you ask one person a question, their answer reflects their individual biases and the limited information they happen to have. If you ask a thousand people the same question and average the answers, random individual errors cancel out and the aggregate is more accurate, provided each person is forming their judgement somewhat independently. Ensemble learning works the same way: the models are the people, the training subsets are the limited information each person has, and the vote or average is the aggregate answer.

The key issue is *independence*. If all your models are trained on exactly the same data and make the same mistakes, averaging their predictions does nothing at all. The diversity between models is what makes ensembles work, and the two families of ensemble methods we cover in this notebook achieve that diversity in quite different ways.

---

## 5. Bagging: Bootstrap Aggregating

🔙 [Back to Table of Contents](#Table-of-Contents)

Bagging was introduced by Leo Breiman in 1996 and the name is a contraction of **Bootstrap Aggregating**. It is the simpler of the two ensemble strategies covered here. It is conceptually simple: train the same algorithm many times on different random subsets of the training data, then aggregate the results.

### 5.1 The Bootstrap

The mechanism that creates diversity between ensemble members is **bootstrap sampling**. Given a training set of $n$ examples, a bootstrap sample is a new dataset of size $n$ drawn **with replacement** from the original. Because sampling is with replacement, some examples will appear more than once, and some will not appear at all.

Statistically, each bootstrap sample is expected to contain approximately $1 - 1/e \approx 63.2\%$ of the unique original examples. The remaining roughly 36.8% of examples are not included in any given bootstrap sample. These left-out examples are called the **out-of-bag** (OOB) sample for that particular tree (more on this in Section 5.3).

The effect is that each model in the ensemble sees a slightly different view of the training data. Some examples appear two or three times and have more influence; others are absent entirely. This is what creates the diversity between models that makes bagging useful.

### 5.2 How Bagging Reduces Variance

To see why averaging helps, consider a collection of $T$ models $h_1, h_2, \ldots, h_T$, each trained on a bootstrap sample. If the individual models each have variance $\sigma^2$ and their errors are **completely independent** of each other (zero correlation), then the variance of the average prediction is:

$$\text{Var}\!\left(\frac{1}{T}\sum_{t=1}^{T} h_t(\mathbf{x})\right) = \frac{\sigma^2}{T}$$

Doubling the number of estimators halves the variance. In theory, with enough models you could reduce variance to zero.

In practice, the models are not perfectly independent: they are all trained on bootstrap samples drawn from the same underlying dataset, so their predictions will be correlated to some degree. The formula generalises to:

$$\text{Var} = \rho \sigma^2 + \frac{1 - \rho}{T} \sigma^2$$

where $\rho$ is the average pairwise correlation between model predictions. As $T \to \infty$, the second term vanishes, but the first term $\rho \sigma^2$ remains. This tells us two things: first, adding more models always helps, up to a point; second, **reducing the correlation between models is just as important as having many of them**. This is the reason that Random Forests (covered in Notebook 6) go a step further than plain bagging by also randomly subsampling features at each split: this extra randomisation further reduces $\rho$ and leads to a stronger variance reduction.

Crucially, this variance reduction does not increase bias. Each individual model in the bag is trained the same way it would be on its own, just on a different data sample. Averaging unbiased (or equally biased) models does not introduce systematic error.

### 5.3 Out-of-Bag Evaluation

Each bootstrap sample omits roughly 36.8% of the training examples. For any given training point, it is out-of-bag for a subset of the trees: the trees that happened not to include it in their bootstrap sample. Those trees have genuinely never seen that point during training, so their predictions on it are a form of held-out evaluation.

By aggregating the out-of-bag predictions for every training point (using only the trees for which that point was out-of-bag), we get a single performance estimate that has the statistical properties of leave-one-out cross-validation, without the cost of running cross-validation explicitly. This is the **OOB score**, available in scikit-learn via `oob_score=True`. It is a computationally cheap and statistically well-grounded alternative to $k$-fold cross-validation when the dataset is large and cross-validation would be slow.

### 5.4 Bagging with scikit-learn

Scikit-learn's `BaggingClassifier` wraps any scikit-learn-compatible classifier as its base estimator and handles the bootstrap sampling and aggregation automatically. The key parameters are:

```python
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),  # the base classifier: any sklearn estimator
    n_estimators=100,                    # how many bootstrap models to train
    max_samples=1.0,                     # fraction (or count) of training rows per bootstrap
    max_features=1.0,                    # fraction (or count) of features per bootstrap
    bootstrap=True,                      # True = sample rows with replacement (bagging)
                                         # False = sample rows without replacement (pasting)
    bootstrap_features=False,            # whether to also resample features with replacement
    oob_score=True,                      # compute the OOB generalisation estimate
    n_jobs=-1,                           # use all available CPU cores
    random_state=42,
)
bag.fit(X_train, y_train)
```

After fitting, the key attributes exposed by the model are:

```python
bag.estimators_          # list of T fitted base classifiers
bag.estimators_samples_  # boolean masks showing which rows each model saw
bag.oob_score_           # OOB accuracy estimate (only if oob_score=True)
bag.oob_decision_function_  # per-sample OOB class probabilities
```

Prediction combines the $T$ individual classifiers by **majority vote** (for `predict`) or by averaging their class probabilities (for `predict_proba`).

### 5.5 Bagging on the Wisconsin Dataset

The Wisconsin Diagnostic Breast Cancer dataset contains 569 tumour samples, each described by 30 numerical features computed from digitised fine-needle aspirate images. The features capture properties of the cell nuclei visible in the image: radius, texture, perimeter, area, smoothness, compactness, concavity, symmetry, and fractal dimension, each summarised by its mean, worst (largest), and standard error across the nuclei present in the image. The classification target is binary: 0 = malignant (212 samples), 1 = benign (357 samples).

This is a well-characterised, cleanly labelled benchmark that is moderately difficult: a single decision tree typically achieves around 90-93% accuracy, while ensemble methods routinely push this above 96%, making the gain from ensembling clearly visible.

The cell below loads the data, applies a stratified 80/20 train/test split, and trains a bagging ensemble of 100 unpruned decision trees. We then compare three things side by side: a single decision tree, a bagging ensemble of 100 trees, and the OOB estimate from the same ensemble.

In [ ]:
# ── Load and split the Wisconsin Breast Cancer dataset ────────────────────────
data = load_breast_cancer()
X, y = data.data, data.target

# Stratified split: preserves the 357:212 benign:malignant ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f'Dataset: {data.target_names[0]} (malignant=0) vs {data.target_names[1]} (benign=1)')
print(f'Total samples : {len(X)} '
      f'({(y==0).sum()} malignant, {(y==1).sum()} benign)')
print(f'Training set  : {len(X_train)} samples')
print(f'Test set      : {len(X_test)} samples')
print(f'Features      : {X.shape[1]}')
print()

# ── Baseline: a single unpruned decision tree ─────────────────────────────────
single_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
single_tree.fit(X_train, y_train)
acc_tree = accuracy_score(y_test, single_tree.predict(X_test))

print(f'Single decision tree  — test accuracy: {acc_tree:.4f}  '
      f'(depth: {single_tree.get_depth()}, leaves: {single_tree.get_n_leaves()})')

# ── Bagging ensemble: 100 unpruned decision trees ─────────────────────────────
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),   # unpruned trees: high variance, low bias
    n_estimators=100,
    max_samples=1.0,                      # each bootstrap sample = same size as training set
    max_features=1.0,                     # use all features (unlike Random Forest)
    bootstrap=True,                       # sample rows with replacement
    oob_score=True,                       # compute OOB generalisation estimate
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
bag.fit(X_train, y_train)
acc_bag = accuracy_score(y_test, bag.predict(X_test))

print(f'Bagging (100 trees)   — test accuracy: {acc_bag:.4f}')
print(f'                        OOB accuracy : {bag.oob_score_:.4f}  '
      f'(estimated without using the test set)')
print()
print(f'Accuracy gain from bagging: {(acc_bag - acc_tree)*100:+.1f} percentage points')

The accuracy gain from bagging is typically 3-5 percentage points on this dataset. The OOB score should be close to the test accuracy, confirming it is a reliable estimator of generalisation performance. Note that the OOB estimate tends to be slightly pessimistic: each bootstrap sample contains only ~63% of unique training points, so the "training set" each tree effectively sees when its OOB accuracy is computed is smaller than the full training set.

The cell below plots a comparison of the two models' confusion matrices and then visualises how test accuracy evolves as we add more estimators to the bag, so you can see the point of diminishing returns.

In [ ]:
# ── Confusion matrices: single tree vs bagging ensemble ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.canvas.header_visible = False
fig.canvas.resizable = True
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'right'

for ax, model, title in zip(
    axes,
    [single_tree, bag],
    ['Single Decision Tree', 'Bagging (100 trees)']
):
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, model.predict(X_test)),
        display_labels=data.target_names,
    ).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontsize=11)

plt.suptitle('Figure 1: Confusion matrices — single tree vs bagging ensemble\n'
             'Wisconsin Breast Cancer dataset', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

# ── Accuracy vs number of estimators ─────────────────────────────────────────
n_range = list(range(1, 201, 5))
train_scores = []
test_scores  = []

for n in n_range:
    m = BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        n_estimators=n,
        max_samples=1.0,
        bootstrap=True,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    m.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, m.predict(X_train)))
    test_scores.append(accuracy_score(y_test, m.predict(X_test)))

fig, ax = plt.subplots(figsize=(8, 4))
fig.canvas.header_visible = False
fig.canvas.resizable = True
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'right'

ax.plot(n_range, train_scores, label='Training accuracy', color='steelblue', lw=1.5)
ax.plot(n_range, test_scores,  label='Test accuracy',     color='tomato',    lw=1.5)
ax.axhline(acc_tree, color='grey', lw=1, ls='--', label=f'Single tree ({acc_tree:.3f})')
ax.set_xlabel('Number of estimators')
ax.set_ylabel('Accuracy')
ax.set_title('Figure 2: Bagging accuracy vs number of trees\nWisconsin dataset', fontsize=11)
ax.legend()
ax.set_ylim(0.88, 1.01)
plt.tight_layout()
plt.show()

The accuracy curve rises steeply at first and then flattens off, reflecting the variance-reduction formula from Section 5.2: the biggest gains come from the first few dozen trees, and adding more beyond around 100 yields only marginal improvement. The training accuracy of individual trees is near-perfect (deep unpruned trees memorise the training set), while the test accuracy is lower for a single tree and converges to a stable plateau as bagging averages away the individual trees' noise.

---

## 6. Boosting: Sequential Error Correction

🔙 [Back to Table of Contents](#Table-of-Contents)

Boosting is a fundamentally different strategy. Rather than training many independent models in parallel and combining them by vote, boosting trains models **sequentially**, with each new model explicitly targeting the mistakes made by all previous models. The result is not an average of equals, but a weighted sum where later models, which are specifically designed to correct the errors of earlier ones, contribute proportionally to how well they address the remaining hard cases.

### 6.1 The Core Idea

The central mechanism in boosting is **adaptive reweighting**. Before training the first model, every training example is assigned equal weight. After training, examples that were misclassified are given higher weight; correctly classified examples are downweighted. The next model is then trained on the reweighted distribution, so it automatically focuses more attention on the hard cases the previous model got wrong. After training again, weights are updated again, and so on.

The final prediction is not a simple majority vote. Each model in the sequence is assigned a **confidence weight** $\alpha_t$ based on how accurate it was. A model that performed barely better than chance gets a small $\alpha_t$; a model that was nearly perfect gets a large one. The final prediction is the sign of the weighted sum:

$$H(\mathbf{x}) = \text{sign}\!\left(\sum_{t=1}^{T} \alpha_t\, h_t(\mathbf{x})\right)$$

where $h_t(\mathbf{x}) \in \{-1, +1\}$ is the prediction of the $t$-th weak learner and $\alpha_t > 0$ is its weight.

The word **weak** in *weak learner* has a precise meaning: it refers to any model that performs at least slightly better than random guessing, that is, its weighted error rate $\epsilon_t < 0.5$. This is a very modest requirement. A decision stump, a tree with only a single split, is the most common choice of weak learner. A stump is almost maximally biased as a standalone classifier, but boosting strings together dozens or hundreds of them and their individual errors are gradually corrected.

### 6.2 Bagging vs Boosting

It is worth pausing to make the comparison explicit before diving into the mechanics of a specific boosting algorithm:

| Property | Bagging | Boosting |
|----------|---------|---------|
| Training order | Parallel: all models train independently | Sequential: each model depends on the previous one |
| Diversity mechanism | Bootstrap sampling: each model sees different data | Adaptive reweighting: each model focuses on previous errors |
| Base learner | Typically high-variance, low-bias (deep trees) | Typically high-bias, low-variance (stumps) |
| What it reduces | Variance | Bias |
| Aggregation | Majority vote or average | Weighted sum of predictions |
| Sensitivity to noise | Generally robust | More sensitive: noisy examples get high weights and can dominate |
| Risk of overfitting | Low (averaging prevents it) | Higher with many rounds, though AdaBoost is surprisingly resistant |
| Parallelisable? | Yes | No, each model must wait for the previous one |

---

## 7. AdaBoost

🔙 [Back to Table of Contents](#Table-of-Contents)

**AdaBoost** (Adaptive Boosting) is the original and most widely studied boosting algorithm, introduced by Freund and Schapire (1997). It won the 2003 Gödel Prize. The core idea, as described above, is to adaptively reweight training examples at each round so that subsequent weak learners focus on what the ensemble has so far found most difficult.

### 7.1 The Algorithm in Detail

The complete AdaBoost procedure for binary classification is as follows. Labels are encoded as $y_i \in \{-1, +1\}$ rather than $\{0, 1\}$, which simplifies the weight-update formula considerably.

**Input:** Training set $\{(\mathbf{x}_i, y_i)\}_{i=1}^{n}$ with $y_i \in \{-1, +1\}$. Number of rounds $T$.

**Initialise:** Set equal weights for all training examples:
$$D_1(i) = \frac{1}{n} \quad \text{for all } i = 1, \ldots, n$$

**For** $t = 1$ to $T$:

1. **Train a weak learner** $h_t$ using the current distribution $D_t$. In practice this means training a classifier with sample weights proportional to $D_t(i)$: points with higher weight have more influence on the model's training objective.

2. **Compute the weighted error** of $h_t$:
   $$\epsilon_t = \sum_{i:\, h_t(\mathbf{x}_i) \neq y_i} D_t(i)$$
   This is not simply the fraction of mistakes: it is the total weight of the misclassified examples. If the high-weight hard cases are mostly wrong, $\epsilon_t$ will be large; if the model gets the hard cases right, $\epsilon_t$ will be small even if it makes many low-weight mistakes.

3. **Compute the learner weight** $\alpha_t$:
   $$\alpha_t = \frac{1}{2} \ln\!\left(\frac{1 - \epsilon_t}{\epsilon_t}\right)$$
   This is a monotonically decreasing function of $\epsilon_t$. When $\epsilon_t \to 0$ (near-perfect), $\alpha_t \to \infty$; when $\epsilon_t = 0.5$ (random guessing), $\alpha_t = 0$; when $\epsilon_t > 0.5$ (worse than chance), $\alpha_t < 0$, meaning the learner's votes are actually inverted. This last case should not occur if the weak learner assumption holds.

4. **Update the sample weights**:
   $$D_{t+1}(i) = \frac{D_t(i) \cdot \exp\!\left(-\alpha_t\, y_i\, h_t(\mathbf{x}_i)\right)}{Z_t}$$
   where $Z_t = \sum_{j} D_t(j) \exp(-\alpha_t y_j h_t(\mathbf{x}_j))$ is a normalisation constant that ensures the weights sum to 1.

   The key is in the exponent $-\alpha_t\, y_i\, h_t(\mathbf{x}_i)$:
   - If $h_t$ correctly classifies example $i$ (i.e., $y_i = h_t(\mathbf{x}_i)$), then $y_i h_t(\mathbf{x}_i) = +1$, so the exponent is $-\alpha_t < 0$, and $e^{-\alpha_t} < 1$. The weight **decreases**: correctly classified points get less attention next round.
   - If $h_t$ misclassifies example $i$ (i.e., $y_i \neq h_t(\mathbf{x}_i)$), then $y_i h_t(\mathbf{x}_i) = -1$, so the exponent is $+\alpha_t > 0$, and $e^{\alpha_t} > 1$. The weight **increases**: misclassified points get more attention next round.

**Output:** The final classifier:
$$H(\mathbf{x}) = \text{sign}\!\left(\sum_{t=1}^{T} \alpha_t\, h_t(\mathbf{x})\right)$$

Each round sharpens the ensemble's attention on the examples it has found hardest so far. The harder a point is to classify (i.e., it keeps being misclassified round after round), the more its weight grows, and the more all subsequent weak learners are nudged to classify it correctly.

### 7.2 The Exponential Loss Connection

AdaBoost can be understood from a different angle: it is equivalent to **forward stagewise additive modelling** under the **exponential loss function**:

$$L(y, f(\mathbf{x})) = \exp(-y \cdot f(\mathbf{x}))$$

where $f(\mathbf{x}) = \sum_{t} \alpha_t h_t(\mathbf{x})$ is the ensemble's raw score and $y \in \{-1, +1\}$ is the true label. The loss is small when $y$ and $f(\mathbf{x})$ have the same sign and large magnitude (confident and correct), and large when they differ in sign (wrong prediction). The algorithm greedily minimises this loss one stage at a time, adding one $(\alpha_t, h_t)$ pair per round without revisiting earlier choices.

This interpretation, due to Friedman, Hastie, and Tibshirani (2000), is important because it explains both AdaBoost's strengths and its weaknesses. The exponential loss penalises misclassified points **exponentially** as a function of the margin $y f(\mathbf{x})$. This is more aggressive than, say, the logistic loss used in logistic regression. It is what drives AdaBoost's rapid convergence on clean data, but it also makes it more sensitive to **outliers and mislabelled examples**: a single example with a very large negative margin can acquire enormous weight and dominate training. On datasets with significant label noise, this is a known failure mode.

### 7.3 AdaBoost with scikit-learn

Scikit-learn's `AdaBoostClassifier` implements the SAMME variant of AdaBoost (the original algorithm generalised cleanly to multi-class settings). The key parameters are:

```python
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # weak learner: a decision stump
    n_estimators=200,     # number of boosting rounds T
    learning_rate=1.0,    # shrinks each alpha_t by this factor (see Section 7.5)
    random_state=42,
)
ada.fit(X_train, y_train)
```

After fitting, the most useful attributes are:

```python
ada.estimators_         # list of T fitted weak learners
ada.estimator_weights_  # the alpha_t value for each round t
ada.estimator_errors_   # the weighted error epsilon_t for each round t
```

A few important notes on the current API:

- The `estimator` parameter (formerly called `base_estimator`, renamed in scikit-learn 1.2) takes any sklearn classifier that supports sample weights in its `fit` method. If `None` is passed, the default is `DecisionTreeClassifier(max_depth=1)`, a decision stump, which is also the most common practical choice.
- The `algorithm` parameter, which previously allowed switching between SAMME and SAMME.R, has been **removed in scikit-learn 1.8**. Only SAMME is now supported. Any code using `algorithm='SAMME'` or `algorithm='SAMME.R'` will raise an error on current versions; the parameter should simply be omitted.

### 7.4 AdaBoost on the Wisconsin Dataset

The cell below trains an AdaBoost ensemble of 200 decision stumps on the same Wisconsin training split used in Section 5.5, then evaluates it on the test set and prints a full classification report.

In [ ]:
# ── AdaBoost on the Wisconsin Breast Cancer dataset ───────────────────────────
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # decision stump: the classic weak learner
    n_estimators=200,
    learning_rate=1.0,
    random_state=RANDOM_STATE,
)
ada.fit(X_train, y_train)
y_pred_ada = ada.predict(X_test)
acc_ada    = accuracy_score(y_test, y_pred_ada)

print('AdaBoost (200 stumps)')
print('=' * 60)
print(classification_report(y_test, y_pred_ada, target_names=data.target_names))

# ── Inspect the per-round diagnostics ─────────────────────────────────────────
errors  = ada.estimator_errors_    # epsilon_t for each round
weights = ada.estimator_weights_   # alpha_t  for each round

print(f'Alpha range : {weights.min():.4f} to {weights.max():.4f}')
print(f'Error range : {errors.min():.4f} to {errors.max():.4f}')
print(f'Final test accuracy: {acc_ada:.4f}')

The classification report shows per-class precision, recall, and F1-score. On the Wisconsin dataset, AdaBoost with 200 stumps typically achieves around 97-98% accuracy, substantially better than a single decision stump (which would achieve around 90%) and comparable to or slightly better than bagging.

Notice the per-round diagnostics: the `estimator_weights_` ($\alpha_t$ values) are higher in early rounds, when the weak learner has a clear signal to exploit and achieves low weighted error. As boosting progresses and the easy cases are handled, the remaining hard cases are more difficult and the weighted error increases, producing smaller $\alpha_t$ values. In the final rounds, the weak learners are contributing relatively little; the bulk of the ensemble's predictive power has already been built up in the earlier rounds.

In [ ]:
# ── Visualise AdaBoost per-round diagnostics ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.canvas.header_visible = False
fig.canvas.resizable = True
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'right'

rounds = np.arange(1, len(errors) + 1)

# Left: weighted error epsilon_t per round
axes[0].plot(rounds, errors, color='tomato', lw=1.5)
axes[0].axhline(0.5, color='grey', ls='--', lw=1, label='Random guessing (ε=0.5)')
axes[0].set_xlabel('Boosting round $t$')
axes[0].set_ylabel('Weighted error $\\epsilon_t$')
axes[0].set_title('Per-round weighted error')
axes[0].legend()
axes[0].set_ylim(-0.01, 0.55)

# Right: learner weight alpha_t per round
axes[1].plot(rounds, weights, color='steelblue', lw=1.5)
axes[1].set_xlabel('Boosting round $t$')
axes[1].set_ylabel('Learner weight $\\alpha_t$')
axes[1].set_title('Per-round learner weight')

plt.suptitle('Figure 3: AdaBoost per-round diagnostics\n'
             'Wisconsin Breast Cancer dataset, 200 stumps', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

The left panel shows that the weighted error in early rounds is well below 0.5 (the random-guessing threshold), meaning each early stump is meaningfully better than chance on the reweighted distribution it sees. As boosting progresses, the error drifts upward: the distribution has been shaped so that the remaining hard cases dominate, and even the best single stump cannot do much better than random on those. The right panel shows the corresponding learner weights: early stumps, which are more accurate, receive higher $\alpha_t$ values and have more influence in the final vote.

### 7.5 Learning Rate and the n_estimators Trade-off

The `learning_rate` parameter (sometimes called the shrinkage factor) multiplies each $\alpha_t$ before it is accumulated into the ensemble:

$$f(\mathbf{x}) = \sum_{t=1}^{T} \nu \cdot \alpha_t\, h_t(\mathbf{x})$$

where $\nu \in (0, 1]$ is the learning rate. Setting $\nu < 1$ makes each round contribute less to the ensemble, which **slows convergence**: you need more rounds ($T$) to reach the same training accuracy. The benefit is that smaller steps are less likely to overfit, particularly on noisy data. There is a direct trade-off between `learning_rate` and `n_estimators`: halving the learning rate roughly doubles the number of rounds needed for the same performance.

In practice, a learning rate between 0.1 and 0.5 combined with a large number of estimators (200-500) often produces better generalisation than the default `learning_rate=1.0` with fewer rounds, particularly when the dataset contains noise. The cell below sweeps learning rate values to show this trade-off directly.

**Note:** the code in the cell below takes about 30 seconds to run.

In [ ]:
# ── Learning rate vs test accuracy sweep ──────────────────────────────────────
learning_rates = [0.05, 0.1, 0.25, 0.5, 1.0]
n_est_range    = list(range(1, 251, 5))

fig, ax = plt.subplots(figsize=(9, 5))
fig.canvas.header_visible = False
fig.canvas.resizable = True
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'right'

colours = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for lr, colour in zip(learning_rates, colours):
    scores = []
    for n in n_est_range:
        m = AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1),
            n_estimators=n,
            learning_rate=lr,
            random_state=RANDOM_STATE,
        )
        m.fit(X_train, y_train)
        scores.append(accuracy_score(y_test, m.predict(X_test)))
    ax.plot(n_est_range, scores, label=f'lr={lr}', color=colour, lw=1.5)

ax.axhline(acc_tree, color='black', lw=1, ls=':', label=f'Single stump baseline')
ax.set_xlabel('Number of estimators')
ax.set_ylabel('Test accuracy')
ax.set_title('Figure 4: AdaBoost test accuracy for different learning rates\n'
             'Wisconsin dataset, decision stump base learner', fontsize=11)
ax.legend(loc='lower right')
ax.set_ylim(0.85, 1.00)
plt.tight_layout()
plt.show()

With `learning_rate=1.0`, the ensemble converges quickly but the test accuracy curve may show more variability. Lower learning rates converge more slowly (the curve rises more gradually) but can achieve comparable or slightly better peak accuracy. This is consistent with the general principle that slower, smaller steps in gradient-based optimisation often generalise better when training is stopped at the right point.

On the Wisconsin dataset, the practical differences between learning rates are small because the data is relatively clean and well-behaved. On nosier datasets the effect is more pronounced.

---

## 8. Comparing Bagging and Boosting Side by Side

🔙 [Back to Table of Contents](#Table-of-Contents)

The cell below runs a systematic comparison of four models on the Wisconsin dataset: a single decision tree, a bagging ensemble, an AdaBoost ensemble with stumps, and an AdaBoost ensemble with slightly deeper trees (max_depth=2), all using the same train/test split. We compare test accuracy, ROC-AUC, and per-class recall.

In [ ]:
# ── Head-to-head comparison on the Wisconsin dataset ─────────────────────────
models = {
    'Single tree (depth unlimited)' : DecisionTreeClassifier(
                                            random_state=RANDOM_STATE),
    'Bagging (100 trees)'           : BaggingClassifier(
                                            estimator=DecisionTreeClassifier(),
                                            n_estimators=100,
                                            oob_score=True,
                                            n_jobs=-1,
                                            random_state=RANDOM_STATE),
    'AdaBoost (200 stumps, lr=1.0)' : AdaBoostClassifier(
                                            estimator=DecisionTreeClassifier(max_depth=1),
                                            n_estimators=200,
                                            learning_rate=1.0,
                                            random_state=RANDOM_STATE),
    'AdaBoost (200 trees d=2, lr=0.5)': AdaBoostClassifier(
                                            estimator=DecisionTreeClassifier(max_depth=2),
                                            n_estimators=200,
                                            learning_rate=0.5,
                                            random_state=RANDOM_STATE),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    results[name] = {
        'Accuracy'        : accuracy_score(y_test, y_pred),
        'ROC-AUC'         : roc_auc_score(y_test, y_prob),
        'Malignant recall': recall_score(y_test, y_pred, pos_label=0),
        'Benign recall'   : recall_score(y_test, y_pred, pos_label=1),
    }

df_results = pd.DataFrame(results).T
print(df_results.to_string(float_format='{:.4f}'.format))

Now to create the comparison ROC curves.

In [ ]:
# ── ROC curves for all four models ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
fig.canvas.header_visible = False
fig.canvas.resizable = True
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'right'

for name, model in models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    RocCurveDisplay.from_predictions(
        y_test, y_prob, name=name, ax=ax,
        plot_chance_level=(name == list(models.keys())[-1]),
    )

ax.set_title('Figure 5: ROC curves — all models\nWisconsin Breast Cancer dataset', fontsize=11)
plt.tight_layout()
plt.show()

Looking at the results table:

- **Accuracy and ROC-AUC** are higher for both ensemble methods than for the single tree, confirming the expected benefit of ensembling.
- **Malignant recall** (how many of the 0 = malignant cases the model catches) is often the most clinically important metric here: failing to flag a malignant tumour as benign is a much more costly error than the reverse. Both ensemble methods tend to improve malignant recall over a single tree.
- **AdaBoost with depth-2 trees** (more expressive weak learners than stumps) often achieves the highest overall accuracy on this dataset, though the differences are small given the dataset's relatively small size and modest class overlap.

The ROC curves show all ensemble models clustering in the top-left corner, confirming that the improvement is consistent across different operating thresholds, not just at the default 0.5 decision threshold. The area under each curve (ROC-AUC) summarises this: values close to 1.0 indicate a classifier that can reliably distinguish malignant from benign tumours across a wide range of sensitivity settings.

The cell below runs a 5-fold stratified cross-validation for a more robust comparison, using the full dataset rather than a single train/test split.

In [ ]:
# ── 5-fold stratified cross-validation ───────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_models = {
    'Single tree'              : DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Bagging (100 trees)'      : BaggingClassifier(
                                        estimator=DecisionTreeClassifier(),
                                        n_estimators=100, n_jobs=-1,
                                        random_state=RANDOM_STATE),
    'AdaBoost (stumps, lr=1.0)': AdaBoostClassifier(
                                        estimator=DecisionTreeClassifier(max_depth=1),
                                        n_estimators=200, learning_rate=1.0,
                                        random_state=RANDOM_STATE),
    'AdaBoost (d=2, lr=0.5)'   : AdaBoostClassifier(
                                        estimator=DecisionTreeClassifier(max_depth=2),
                                        n_estimators=200, learning_rate=0.5,
                                        random_state=RANDOM_STATE),
}

print('5-fold stratified cross-validation (full dataset)\n')
print(f'{"Model":<35} {"Mean accuracy":>14}  {"Std":>6}')
print('-' * 60)

for name, model in cv_models.items():
    cv_scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    print(f'{name:<35} {cv_scores.mean():.4f}  ±  {cv_scores.std():.4f}')

Cross-validation gives a more stable estimate of generalisation performance by averaging over five different train/test splits rather than relying on one. The standard deviations show how much performance varies across folds: ensemble methods typically show lower variance here as well, since their predictions are already an internal average.

---

## 9. Summary

🔙 [Back to Table of Contents](#Table-of-Contents)

| Concept | Description |
|---------|-------------|
| **Ensemble learning** | Combining predictions from multiple models to produce a prediction that is more accurate or more stable than any single model alone. |
| **Bias-variance decomposition** | Total expected error = bias² + variance + irreducible noise. Ensembles address bias (boosting) or variance (bagging) depending on the strategy. |
| **Weak learner** | Any classifier that performs at least slightly better than random guessing ($\epsilon_t < 0.5$). Decision stumps are the canonical example. |
| **Bootstrap sampling** | Sampling $n$ examples with replacement from a dataset of size $n$. Each bootstrap sample includes approximately 63.2% unique examples; the rest are out-of-bag. |
| **Out-of-bag (OOB) score** | A generalisation error estimate computed from the predictions of trees for which a given point was not included in the bootstrap. Available via `oob_score=True`. Statistically equivalent to leave-one-out cross-validation, without the computational cost. |
| **Bagging** (Breiman, 1996) | Train $T$ models in parallel on bootstrap samples; aggregate by majority vote or probability averaging. Reduces variance without increasing bias. Best applied to high-variance base learners such as unpruned decision trees. |
| **`BaggingClassifier`** | scikit-learn class implementing bagging. Key params: `estimator`, `n_estimators`, `max_samples`, `max_features`, `bootstrap`, `oob_score`. |
| **Boosting** | Train $T$ weak learners sequentially, with each one focusing on the examples misclassified by its predecessors. Reduces bias. Not parallelisable. More sensitive to noise than bagging. |
| **Adaptive reweighting** | After each round of boosting, misclassified examples have their weight increased and correctly classified ones have their weight decreased, so the next learner is forced to focus on the hard cases. |
| **AdaBoost** (Freund & Schapire, 1997) | The canonical boosting algorithm. Learner weight $\alpha_t = \frac{1}{2}\ln\!\left(\frac{1-\epsilon_t}{\epsilon_t}\right)$. Weight update: $D_{t+1}(i) \propto D_t(i)\exp(-\alpha_t y_i h_t(\mathbf{x}_i))$. Final prediction: $\text{sign}(\sum_t \alpha_t h_t(\mathbf{x}))$. Minimises exponential loss. |
| **`AdaBoostClassifier`** | scikit-learn class implementing AdaBoost (SAMME). Key params: `estimator`, `n_estimators`, `learning_rate`. Note: the `algorithm` parameter was removed in scikit-learn 1.8. |
| **Learning rate** | Shrinks each $\alpha_t$ by a factor $\nu \in (0,1]$ before accumulation. Smaller values require more estimators but can generalise better, especially on noisy data. |
| **Exponential loss** | The loss function implicitly minimised by AdaBoost: $L(y, f) = e^{-yf(\mathbf{x})}$. Penalises misclassified points exponentially; makes AdaBoost fast on clean data but sensitive to outliers and label noise. |

---

---

## 10. References

🔙 [Back to Table of Contents](#Table-of-Contents)

Breiman, L. (1996). Bagging predictors. *Machine Learning*, 24(2), 123–140. https://doi.org/10.1007/BF00058655

Freund, Y., & Schapire, R. E. (1997). A decision-theoretic generalization of on-line learning and an application to boosting. *Journal of Computer and System Sciences*, 55(1), 119–139. https://doi.org/10.1006/jcss.1997.1504

Friedman, J., Hastie, T., & Tibshirani, R. (2000). Additive logistic regression: A statistical view of boosting. *The Annals of Statistics*, 28(2), 337–407. https://doi.org/10.1214/aos/1016218223

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830. http://jmlr.org/papers/v12/pedregosa11a.html

Street, W. N., Wolberg, W. H., & Mangasarian, O. L. (1993). Nuclear feature extraction for breast tumor diagnosis. *SPIE Proceedings: Biomedical Image Processing and Biomedical Visualization*, 1905, 861–870. https://doi.org/10.1117/12.148698